1. Importing packages and creating folders

In [1]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

raw_dir = Path("data/subset_raw")
clean_dir = Path("data/clean")
report_dir = Path("data/fastp_reports")
reference_dir = Path("data/reference")
quant_dir = Path("data/quant")
results_dir = Path("results")

for folder in [
    raw_dir,
    clean_dir,
    report_dir,
    reference_dir,
    quant_dir,
    results_dir,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


2. Retrieving the metadata from the 12 samples

TSV file with TPM valus 

In [2]:
ena_url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    "?accession=PRJNA1194989"
    "&result=read_run"
    "&fields=run_accession,sample_accession,experiment_title,library_layout"
    "&format=tsv"
)

samples = pd.read_csv(ena_url, sep="\t")

samples["sample_name"] = samples["experiment_title"].str.extract(
    r"GSM\d+:\s+(\S+)"
)

samples["condition"] = (
    samples["sample_name"]
    .str.replace(r"_\d+$", "", regex=True)
    .str.replace("624Mel_", "", regex=False)
)

samples["replicate"] = (
    samples["sample_name"]
    .str.extract(r"_(\d+)$")[0]
    .astype(int)
)

samples = samples.sort_values(
    ["condition", "replicate"]
).reset_index(drop=True)

display(
    samples[
        [
            "run_accession",
            "sample_name",
            "condition",
            "replicate",
            "library_layout",
        ]
    ]
)

print("Number of samples:", len(samples))

,run_accession,sample_name,condition,replicate,library_layout
0,SRR31631262,624Mel_siCTRL_1,siCTRL,1,PAIRED
1,SRR31631258,624Mel_siCTRL_2,siCTRL,2,PAIRED
2,SRR31631254,624Mel_siCTRL_3,siCTRL,3,PAIRED
3,SRR31631261,624Mel_siCTRL+IFNg_1,siCTRL+IFNg,1,PAIRED
4,SRR31631257,624Mel_siCTRL+IFNg_2,siCTRL+IFNg,2,PAIRED
5,SRR31631253,624Mel_siCTRL+IFNg_3,siCTRL+IFNg,3,PAIRED
6,SRR31631260,624Mel_siMITF_1,siMITF,1,PAIRED
7,SRR31631256,624Mel_siMITF_2,siMITF,2,PAIRED
8,SRR31631252,624Mel_siMITF_3,siMITF,3,PAIRED
9,SRR31631259,624Mel_siMITF+IFNg_1,siMITF+IFNg,1,PAIRED


Number of samples: 12


3. Downloading 1000000 reads from all 12 samples

SRA files 

In [3]:
max_spots = 1_000_000

for run in samples["run_accession"]:
    read1 = raw_dir / f"{run}_1.fastq.gz"
    read2 = raw_dir / f"{run}_2.fastq.gz"

    if read1.exists() and read2.exists():
        print(f"{run}: files already exist — skipping")
        continue

    print(f"Retrieving {max_spots:,} paired spots from {run}...")

    subprocess.run(
        [
            "fastq-dump",
            "-N", "1",
            "-X", str(max_spots),
            "--split-files",
            "--gzip",
            "--outdir", str(raw_dir),
            run,
        ],
        check=True,
    )

print("Subset retrieval complete.")

Retrieving 1,000,000 paired spots from SRR31631262...
Read 1000000 spots for SRR31631262
Written 1000000 spots for SRR31631262
Retrieving 1,000,000 paired spots from SRR31631258...
Read 1000000 spots for SRR31631258
Written 1000000 spots for SRR31631258
Retrieving 1,000,000 paired spots from SRR31631254...
Read 1000000 spots for SRR31631254
Written 1000000 spots for SRR31631254
Retrieving 1,000,000 paired spots from SRR31631261...
Read 1000000 spots for SRR31631261
Written 1000000 spots for SRR31631261
Retrieving 1,000,000 paired spots from SRR31631257...
Read 1000000 spots for SRR31631257
Written 1000000 spots for SRR31631257
Retrieving 1,000,000 paired spots from SRR31631253...
Read 1000000 spots for SRR31631253
Written 1000000 spots for SRR31631253
Retrieving 1,000,000 paired spots from SRR31631260...
Read 1000000 spots for SRR31631260
Written 1000000 spots for SRR31631260
Retrieving 1,000,000 paired spots from SRR31631256...
Read 1000000 spots for SRR31631256
Written 1000000 spots 

4. Confirming the raw FASTQs; 24 in total, one reverse and one forward FASTQ from each sample

In [4]:
raw_fastqs = sorted(raw_dir.glob("*.fastq.gz"))

print("Raw FASTQ files:", len(raw_fastqs))

assert len(samples) == 12, "Expected 12 samples."
assert len(raw_fastqs) == 24, "Expected 24 paired-end FASTQ files."

for run in samples["run_accession"]:
    assert (raw_dir / f"{run}_1.fastq.gz").exists()
    assert (raw_dir / f"{run}_2.fastq.gz").exists()

print("All 12 paired-end samples are present.")

Raw FASTQ files: 24
All 12 paired-end samples are present.


5. Using FASTP to clean pair end reads; resulting in clean FASTQs and QC reports.

In [5]:
for run in samples["run_accession"]:
    input_r1 = raw_dir / f"{run}_1.fastq.gz"
    input_r2 = raw_dir / f"{run}_2.fastq.gz"

    output_r1 = clean_dir / f"{run}_1.clean.fastq.gz"
    output_r2 = clean_dir / f"{run}_2.clean.fastq.gz"

    json_report = report_dir / f"{run}.fastp.json"
    html_report = report_dir / f"{run}.fastp.html"

    if (
        output_r1.exists()
        and output_r2.exists()
        and json_report.exists()
        and html_report.exists()
    ):
        print(f"{run}: already cleaned — skipping")
        continue

    print(f"Cleaning {run}...")

    subprocess.run(
        [
            "fastp",
            "--in1", str(input_r1),
            "--in2", str(input_r2),
            "--out1", str(output_r1),
            "--out2", str(output_r2),
            "--json", str(json_report),
            "--html", str(html_report),
            "--detect_adapter_for_pe",
            "--thread", "4",
        ],
        check=True,
    )

print("Cleaning complete.")

Cleaning SRR31631262...


Detecting adapter sequence for read1...
>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123312436(98.6499%)
Q30 bases: 120357370(96.2859%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122890277(98.3122%)
Q30 bases: 119404394(95.5235%)

Read1 after filtering:
total reads: 991435
total bases: 123501739
Q20 bases: 122079022(98.848%)
Q30 bases: 119241095(96.5501%)

Read2 after filtering:
total reads: 991435
total bases: 123502701
Q20 bases: 121939913(98.7346%)
Q30 bases: 118621518(96.0477%)

Filtering result:
reads passed filter: 1982870
reads failed due to low quality: 16636
reads failed due to too many N: 0
reads failed due to too short: 494
reads with adapter trimmed: 49237
bases trimmed due to adapters: 903889

Duplication rate: 28.4395%

Insert size

Cleaning SRR31631258...


>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123368927(98.6951%)
Q30 bases: 120583393(96.4667%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122622187(98.0977%)
Q30 bases: 119036618(95.2293%)

Read1 after filtering:
total reads: 989461
total bases: 123204939
Q20 bases: 121863988(98.9116%)
Q30 bases: 119206507(96.7546%)

Read2 after filtering:
total reads: 989461
total bases: 123207430
Q20 bases: 121496045(98.611%)
Q30 bases: 118118618(95.8697%)

Filtering result:
reads passed filter: 1978922
reads failed due to low quality: 20668
reads failed due to too many N: 0
reads failed due to too short: 410
reads with adapter trimmed: 53044
bases trimmed due to adapters: 1001711

Duplication rate: 25.7227%

Insert size peak (evaluated by paired-end reads): 

Cleaning SRR31631254...


...

>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123202768(98.5622%)
Q30 bases: 120300704(96.2406%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122874726(98.2998%)
Q30 bases: 119508695(95.607%)

Read1 after filtering:
total reads: 990782
total bases: 123379552
Q20 bases: 121898709(98.7998%)
Q30 bases: 119129569(96.5554%)

Read2 after filtering:
total reads: 990782
total bases: 123381690
Q20 bases: 121814296(98.7296%)
Q30 bases: 118624194(96.1441%)

Filtering result:
reads passed filter: 1981564
reads failed due to low quality: 17976
reads failed due to too many N: 0
reads failed due to too short: 460
reads with adapter trimmed: 50846
bases trimmed due to adapters: 984198

Duplication rate: 23.4104%

Insert size peak (evaluated by paired-end reads): 1

Cleaning SRR31631251...


>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123267776(98.6142%)
Q30 bases: 120327878(96.2623%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123072781(98.4582%)
Q30 bases: 119808620(95.8469%)

Read1 after filtering:
total reads: 992142
total bases: 123620956
Q20 bases: 122185472(98.8388%)
Q30 bases: 119365602(96.5577%)

Read2 after filtering:
total reads: 992142
total bases: 123622776
Q20 bases: 122159665(98.8165%)
Q30 bases: 119045497(96.2974%)

Filtering result:
reads passed filter: 1984284
reads failed due to low quality: 15312
reads failed due to too many N: 0
reads failed due to too short: 404
reads with adapter trimmed: 40429
bases trimmed due to adapters: 840361

Duplication rate: 28.5831%

Insert size peak (evaluated by paired-end reads): 

Cleaning complete.



JSON report: data/fastp_reports/SRR31631251.fastp.json
HTML report: data/fastp_reports/SRR31631251.fastp.html

fastp --in1 data/subset_raw/SRR31631251_1.fastq.gz --in2 data/subset_raw/SRR31631251_2.fastq.gz --out1 data/clean/SRR31631251_1.clean.fastq.gz --out2 data/clean/SRR31631251_2.clean.fastq.gz --json data/fastp_reports/SRR31631251.fastp.json --html data/fastp_reports/SRR31631251.fastp.html --detect_adapter_for_pe --thread 4 
fastp v0.23.4, time used: 9 seconds


Cheack what before and after fastp

In [7]:
report_dir = Path("data/fastp_reports")

rows = []

for report_file in sorted(report_dir.glob("*.json")):
    with open(report_file) as f:
        report = json.load(f)

    before = report["summary"]["before_filtering"]
    after = report["summary"]["after_filtering"]

    rows.append({
        "Sample": report_file.stem.replace(".fastp", ""),
        "Reads before fastp": before["total_reads"],
        "Reads after fastp": after["total_reads"],
        "Reads retained (%)": 100 * after["total_reads"] / before["total_reads"],
    })

fastp_summary = pd.DataFrame(rows)

fastp_summary.round(2)

,Sample,Reads before fastp,Reads after fastp,Reads retained (%)
0,SRR31631251,2000000,1984284,99.21
1,SRR31631252,2000000,1982860,99.14
2,SRR31631253,2000000,1971558,98.58
3,SRR31631254,2000000,1984034,99.20
4,SRR31631255,2000000,1981564,99.08
5,SRR31631256,2000000,1983152,99.16
6,SRR31631257,2000000,1981878,99.09
7,SRR31631258,2000000,1978922,98.95
8,SRR31631259,2000000,1977388,98.87
9,SRR31631260,2000000,1982960,99.15


Get index for kallisto 

In [8]:
%%bash

wget https://github.com/pachterlab/kallisto/releases/download/v0.52.0/kallisto_linux-v0.52.0.tar.gz --quiet
wget https://github.com/pachterlab/kallisto-transcriptome-indices/releases/download/v1/human_index_standard.tar.xz --quiet

tar zxvf kallisto_linux-v0.52.0.tar.gz
tar xf human_index_standard.tar.xz

kallisto/
kallisto/test/
kallisto/README.md
kallisto/._kallisto
kallisto/kallisto
kallisto/._license.txt
kallisto/license.txt
kallisto/test/reads_1.fastq.gz
kallisto/test/transcripts.fasta.gz
kallisto/test/sc_reads_2.fastq.gz
kallisto/test/README.md
kallisto/test/._chrom.txt
kallisto/test/chrom.txt
kallisto/test/Snakefile
kallisto/test/reads_2.fastq.gz
kallisto/test/transcripts.gtf.gz
kallisto/test/sc_reads_1.fastq.gz


Try kallisto for 1 sample 

In [10]:
%%bash

time kallisto/kallisto quant \
  -i index.idx \
  data/clean/SRR31631251_1.clean.fastq.gz \
  data/clean/SRR31631251_2.clean.fastq.gz \
  -o testing \
  -t 2


[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631251_1.clean.fastq.gz
                             data/clean/SRR31631251_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 992,142 reads, 869,926 reads pseudoaligned
[quant] estimated average fragment length: 280.145
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 898 rounds


real	1m5.300s
user	1m29.207s
sys	0m2.932s


Kallisto for all the 12 samples 

In [11]:
%%bash

for r1 in data/clean/*_1.clean.fastq.gz
do
    run=$(basename "$r1" _1.clean.fastq.gz)
    r2="data/clean/${run}_2.clean.fastq.gz"

    echo "Running kallisto for $run"

    kallisto/kallisto quant \
        -i index.idx \
        "$r1" \
        "$r2" \
        -o "data/quant/$run" \
        -t 2
done

Running kallisto for SRR31631251



[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631251_1.clean.fastq.gz
                             data/clean/SRR31631251_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 992,142 reads, 869,926 reads pseudoaligned
[quant] estimated average fragment length: 280.145
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 898 rounds



Running kallisto for SRR31631252



[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631252_1.clean.fastq.gz
                             data/clean/SRR31631252_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 991,430 reads, 870,792 reads pseudoaligned
[quant] estimated average fragment length: 278.292
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 959 rounds



Running kallisto for SRR31631253


...


[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631260_1.clean.fastq.gz
                             data/clean/SRR31631260_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 991,480 reads, 855,190 reads pseudoaligned
[quant] estimated average fragment length: 262.472
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 939 rounds



Running kallisto for SRR31631261



[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631261_1.clean.fastq.gz
                             data/clean/SRR31631261_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 991,068 reads, 819,343 reads pseudoaligned
[quant] estimated average fragment length: 271.553
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 965 rounds



Running kallisto for SRR31631262



[quant] fragment length distribution will be estimated from the data
[index] k-mer length: 31
[index] number of targets: 227,665
[index] number of k-mers: 139,900,295
[index] number of D-list k-mers: 5,477,475
[quant] running in paired-end mode
[quant] will process pair 1: data/clean/SRR31631262_1.clean.fastq.gz
                             data/clean/SRR31631262_2.clean.fastq.gz
[quant] finding pseudoalignments for the reads ... done
[quant] processed 991,435 reads, 868,064 reads pseudoaligned
[quant] estimated average fragment length: 269.716
[   em] quantifying the abundances ... done
[   em] the Expectation-Maximization algorithm ran for 915 rounds

